# Assignment 6: Group Convolutions

In past assignments, we already tackled equivariance for GNNs. Today we will look at equivariance for CNNs. Standard CNNs are already translation-equivariant on $\mathbb{Z}^2$ by construction, so as a next step, we will add equivariance to a rotation group today. We focus on the **p4** wallpaper group — the semidirect product $p4 = C_4 \ltimes \mathbb{Z}^2$ of 90° rotations and integer translations — and benchmark a $p4$-equivariant CNN ([Cohen & Welling, 2016](https://arxiv.org/abs/1602.07576)) against two simpler baselines.

**Roadmap.** We tackle a 10-class classification problem on randomly-rotated FashionMNIST with three models, ordered by how much rotational structure each one bakes in:

1. **Vanilla CNN** — translation-equivariant only; no rotation prior at all.
2. **Isotropic CNN** — each filter is SO(2)-symmetric so rotation invariance happens *per filter*.
3. **p4-CNN** — translation- *and* 90°-rotation-equivariant as a whole architecture.


In [ ]:
import math
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

torch.manual_seed(42)
np.random.seed(42)


<div class="alert alert-info">
    <h3>Utility: shared helpers</h3>
    Nothing to do here, just run the cell. We define three helpers used throughout:
    <ul>
        <li><code>train_model</code> — a generic training / evaluation loop.</li>
        <li><code>invariance_error</code> — measures how far a model deviates from being rotation-invariant at a given angle, averaged over a batch.</li>
        <li><code>imshow_grid</code> — visualizes a batch of feature maps in a 1xN grid (used for the lifted p4 feature maps).</li>
    </ul>
</div>


In [ ]:
def train_model(model, train_loader, test_loader, epochs=15, lr=1e-3, log_every=1):
    '''Generic training loop. Returns final test accuracy.'''
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for ep in range(epochs):
        model.train()
        total, correct, total_loss = 0, 0, 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = F.cross_entropy(logits, y)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total_loss += loss.item() * x.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            total += x.size(0)
        if (ep + 1) % log_every == 0:
            print(f"  epoch {ep+1:2d} | train loss {total_loss/total:.4f} | train acc {correct/total:.3f}")
    # final test accuracy
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in test_loader:
            x, y = x.to(device), y.to(device)
            correct += (model(x).argmax(1) == y).sum().item()
            total += x.size(0)
    acc = correct / total
    print(f"  test accuracy: {acc:.3f}")
    return acc


def invariance_error(model, x, angle_deg):
    '''Max abs deviation between f(R x) and f(x) on a batch, where R rotates by angle_deg.
    For a perfectly rotation-invariant model the error is 0 for any angle.
    We use this as a quantitative probe of how rotation-robust each architecture is.'''
    model.eval()
    with torch.no_grad():
        x = x.to(device)
        x_rot = transforms.functional.rotate(x, angle_deg)
        out = model(x)
        out_rot = model(x_rot)
        return (out - out_rot).abs().max().item()


def imshow_grid(maps, title=""):
    '''maps: tensor of shape (N, H, W) -- plots N panels in a row.'''
    n = maps.shape[0]
    fig, axes = plt.subplots(1, n, figsize=(2.2 * n, 2.4))
    if n == 1:
        axes = [axes]
    vmax = maps.abs().max().item()
    for i, ax in enumerate(axes):
        ax.imshow(maps[i].cpu().numpy(), cmap="RdBu_r", vmin=-vmax, vmax=vmax)
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_title(f"r={i}")
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


# Problem setup

We design the task so that **rotation appears only at test time**, and only at the angles the $p4$ group contains — multiples of 90°. The training images are upright; each test image is rotated by a uniformly-random $k \cdot 90°$ for $k \in \{0, 1, 2, 3\}$. This is exactly the symmetry group our equivariant network will be built around, and it's the cleanest setting to compare architectures: a vanilla CNN has no reason to behave consistently under these rotations, while an architecturally $p4$-equivariant network is guaranteed to.


<div class="alert alert-info">
    <h3>Provided: FashionMNIST loaders (upright train, 90°-rotated test)</h3>
    Nothing to do here, just run the cell. The training images are kept upright; each test image is rotated by a uniformly-random multiple of 90° via the <code>Random90</code> transform. We subsample to 10000 train / 5000 test images and use a batch size of 64.
</div>


In [ ]:
DATA_ROOT = "./data"

class Random90:
    '''Rotate a tensor image by a uniformly-random multiple of 90 degrees.'''
    def __call__(self, x):
        k = int(torch.randint(0, 4, (1,)).item())
        return torch.rot90(x, k=k, dims=(-2, -1))

train_transform = transforms.ToTensor()
test_transform  = transforms.Compose([
    transforms.ToTensor(),
    Random90(),
])

train_full = datasets.FashionMNIST(DATA_ROOT, train=True,  download=True, transform=train_transform)
test_full  = datasets.FashionMNIST(DATA_ROOT, train=False, download=True, transform=test_transform)

g = torch.Generator().manual_seed(0)
train_idx = torch.randperm(len(train_full), generator=g)[:10000]
test_idx  = torch.randperm(len(test_full),  generator=g)[:5000]
train_set = Subset(train_full, train_idx.tolist())
test_set  = Subset(test_full,  test_idx.tolist())

train_loader = DataLoader(train_set, batch_size=64, shuffle=True,  num_workers=0)
test_loader  = DataLoader(test_set,  batch_size=64, shuffle=False, num_workers=0)

print(f"train: {len(train_set)} (upright) | test: {len(test_set)} (rotated by k*90°)")


<div class="alert alert-info">
    <h3>Visualize the train and test splits side-by-side</h3>
    Nothing to do here, just run the cell. The top row shows upright training samples; the bottom row shows the rotated test samples the model will be evaluated on.
</div>


In [ ]:
x_tr, y_tr = next(iter(train_loader))
x_te, y_te = next(iter(test_loader))
class_names = ["T-shirt","Trouser","Pullover","Dress","Coat","Sandal","Shirt","Sneaker","Bag","Boot"]

fig, axes = plt.subplots(2, 6, figsize=(9, 3.2))
for i in range(6):
    axes[0, i].imshow(x_tr[i, 0].numpy(), cmap="gray")
    axes[0, i].set_title(class_names[y_tr[i].item()], fontsize=8)
    axes[0, i].axis("off")
    axes[1, i].imshow(x_te[i, 0].numpy(), cmap="gray")
    axes[1, i].set_title(class_names[y_te[i].item()], fontsize=8)
    axes[1, i].axis("off")
axes[0, 0].set_ylabel("train (upright)", fontsize=9)
axes[1, 0].set_ylabel("test (rotated)",  fontsize=9)
fig.suptitle("FashionMNIST: upright train vs. rotated test")
plt.tight_layout()
plt.show()


# 1. VanillaCNN as a baseline

The plain convolutional layer is **translation-equivariant** but has no rotational structure built in. Trained only on upright clothing, a vanilla CNN has no reason to behave consistently on rotated inputs — we should expect a substantial drop in test accuracy, plus large invariance errors. Before training one, let's see the root of the problem at the level of a single filter.


<div class="alert alert-warning">
    <h3>Task 1.1: A single conv filter is not <span>$C_4$</span>-equivariant [1 pt]</h3>
    Let's see why a vanilla CNN doesn't help us with rotations. Pick one image from the trainloader and initialize a single random <code>nn.Conv2d</code>. Then, compute the following two feature maps:
    <ul>
        <li>Apply the filter to the 90°-rotated image, and</li>
        <li>Apply the filter to the unrotated image and rotate the output afterwards.</li>
    </ul>
    Plot both feature maps side-by-side and interpret what you see. What <i>should</i> the two panels look like if the convolution were $C_4$-equivariant?
</div>


In [ ]:
### BEGIN SOLUTION

### END SOLUTION


### Interpretation

...


<div class="alert alert-info">
    <h3>Provided: VanillaCNN training</h3>
    Nothing to do here, just run the cell. We define a standard 3-layer CNN, train it for 15 epochs on the upright training set, and report the test accuracy on the rotated test set. This becomes the baseline we'll later compare the isotropic and $p4$-equivariant networks against.
</div>


In [ ]:
class VanillaCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 48, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(48, 96, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(96, 96, kernel_size=3, padding=1)
        self.fc = nn.Linear(192, 10)        # 192 = 96 channels x (mean, max)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = F.relu(self.conv3(x))
        # global readout: concatenate spatial mean and spatial max
        x = torch.cat([x.mean(dim=(-2, -1)), x.amax(dim=(-2, -1))], dim=-1)
        return self.fc(x)


vanilla = VanillaCNN()
print(f"VanillaCNN params: {sum(p.numel() for p in vanilla.parameters()):,}")

print("\nTraining VanillaCNN ...")
acc_vanilla = train_model(vanilla, train_loader, test_loader, epochs=15)


<div class="alert alert-info">
    <h3>Invariance error vs angle</h3>
    Nothing to do — we probe the trained model at several rotation angles and record the max output deviation.
</div>


In [ ]:
ANGLES = [0, 90, 180, 270, 360]
x_probe, _ = next(iter(test_loader))

err_vanilla = [invariance_error(vanilla, x_probe, a) for a in ANGLES]
for a, e in zip(ANGLES, err_vanilla):
    print(f"VanillaCNN | angle {a:3d}° | max output deviation = {e:.3f}")


# 2. Baseline: the isotropic CNN

A way to obtain *rotation-equivariance* for free is to constrain each filter to be SO(2)-symmetric — i.e. its weights depend only on the distance from the centre.

For a $3\times3$ kernel discretized on a pixel grid we approximate this with **two** learnable scalars per filter:

- $w_c$ — the centre weight
- $w_r$ — the shared "ring" weight applied to the 8 neighbouring pixels

So an isotropic $3\times3$ filter looks like

$$
\psi \;=\; \begin{bmatrix} w_r & w_r & w_r \\ w_r & w_c & w_r \\ w_r & w_r & w_r \end{bmatrix}.
$$

Because $\psi$ is invariant under any 90° rotation, *each* filter response is rotation-invariant on its own.


<div class="alert alert-warning">
    <h3>Task 2.1: IsotropicConv2d [1 pt]</h3>
    Implement a custom layer with two learnable parameter tensors of shape <code>(out_channels, in_channels)</code>:
    <ul>
        <li><code>w_center</code> — the centre weight for every (out, in) pair</li>
        <li><code>w_ring</code> — the shared ring weight</li>
    </ul>
    In <code>forward</code>, assemble the full <code>(out_channels, in_channels, 3, 3)</code> weight tensor on the fly and call <code>F.conv2d</code> with <code>padding=1</code>. Include a learnable bias of shape <code>(out_channels,)</code>.
    <br><br>
    <i>Hint (initialization):</i> use a Kaiming-style uniform init (standard for convolutional layers in torch) with <code>bound = 1 / sqrt(in_channels * 9)</code>. The factor 9 reflects the receptive field area of the assembled $3\times3$ filter (even though only 2 entries are learnable, the kernel still reads 9 input cells, and we want the per-output variance to roughly match a standard <code>nn.Conv2d</code>).
</div>


In [ ]:
class IsotropicConv2d(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        ### BEGIN SOLUTION

        ### END SOLUTION

    def assemble_weight(self):
        ### BEGIN SOLUTION

        ### END SOLUTION

    def forward(self, x):
        ### BEGIN SOLUTION

        ### END SOLUTION


# Sanity check: a single isotropic filter is rotation-invariant.
iso_layer = IsotropicConv2d(1, 4)
w = iso_layer.assemble_weight()
for k in range(4):
    assert torch.allclose(w, torch.rot90(w, k=k, dims=(-2, -1)), atol=1e-6), \
        "Isotropic kernel should be invariant under 90° rotations."
print("Isotropic filter is invariant under {0°, 90°, 180°, 270°} rotations.")


<div class="alert alert-warning">
    <h3>Task 2.2: An isotropic filter <i>is</i> $C_4$-equivariant [0.5 pt]</h3>
    Repeat the visualization from Task 1.1, but now with an <code>IsotropicConv2d</code> layer in place of <code>nn.Conv2d</code>. Pick one image <code>x</code> from the training loader and
    <ul>
        <li>Apply the filter to the 90°-rotated image, and</li>
        <li>Apply the filter to the unrotated image and rotate the output afterwards.</li>
    </ul>
    Plot both feature maps side-by-side. What's different from Task 1.1?
</div>


In [ ]:
### BEGIN SOLUTION

### END SOLUTION


### Interpretation

...

<div class="alert alert-warning">
    <h3>Task 2.3: Downside of the isotropic filter [0.5 pt]</h3>
    We gained SO(2) symmetry for free — but at what cost? Answer in 2–3 sentences:
    <ul>
        <li>How many <b>learnable parameters per filter</b> does <code>IsotropicConv2d</code> have, compared to a standard <code>nn.Conv2d</code> with a $3\times3$ kernel?</li>
        <li>What kind of patterns can a standard $3\times3$ filter detect that an isotropic filter <i>cannot</i>?</li>
    </ul>
</div>


### Interpretation

...


<div class="alert alert-info">
    <h3>Provided: IsotropicCNN training</h3>
    Nothing to do here, just run the cell. The <code>IsotropicCNN</code> is literally a "constrained vanilla CNN": the same architecture and channel widths as <code>VanillaCNN</code> — <b>(48, 96, 96)</b>, mean+max readout — but with <code>IsotropicConv2d</code> in place of <code>nn.Conv2d</code>, so every filter is forced to be SO(2)-symmetric. Trained for 15 epochs.
</div>


In [ ]:
class IsotropicCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = IsotropicConv2d(1, 48)
        self.conv2 = IsotropicConv2d(48, 96)
        self.conv3 = IsotropicConv2d(96, 96)
        self.fc = nn.Linear(192, 10)        # 192 = 96 channels x (mean, max)

    def forward(self, x):
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)
        x = F.relu(self.conv3(x))
        x = torch.cat([x.mean(dim=(-2, -1)), x.amax(dim=(-2, -1))], dim=-1)
        return self.fc(x)


iso_net = IsotropicCNN()
print(f"IsotropicCNN params: {sum(p.numel() for p in iso_net.parameters()):,}")

print("\nTraining IsotropicCNN ...")
acc_iso = train_model(iso_net, train_loader, test_loader, epochs=15)


<div class="alert alert-info">
    <h3>Invariance error vs angle</h3>
    Nothing to do — we probe the trained model at several rotation angles and record the max output deviation.
</div>


In [ ]:
err_iso = [invariance_error(iso_net, x_probe, a) for a in ANGLES]
for a, e in zip(ANGLES, err_iso):
    print(f"IsotropicCNN | angle {a:3d}° | max output deviation = {e:.3f}")


# 3. The p4 group

We now upgrade from per-filter to per-architecture equivariance. The relevant group is the wallpaper group $p4$, the semidirect product

$$
p4 \;=\; C_4 \ltimes \mathbb{Z}^2,
$$

where $C_4 = \{e, r, r^2, r^3\}$ is the cyclic group of 90° rotations and $\mathbb{Z}^2$ is the lattice of integer translations. An element of $p4$ is a pair $g = (r^k, \mathbf{t})$ with rotation index $k \in \{0, 1, 2, 3\}$ and translation $\mathbf{t} \in \mathbb{Z}^2$. The group product is

$$
(r^{k_1}, \mathbf{t}_1)\,(r^{k_2}, \mathbf{t}_2) \;=\; \bigl(r^{(k_1 + k_2)\bmod 4},\; \mathbf{t}_1 + r^{k_1}\mathbf{t}_2\bigr).
$$

**Group actions.**
The group $p4$ acts directly on the **domain** $\mathbb{Z}^2$: an element $g = (r^k, \mathbf{t})$ maps a pixel $\mathbf{u}$ to $g.\mathbf{u} = r^k\mathbf{u} + \mathbf{t}$. An image, however, is not a point of the domain but a **function** on it, $x : \mathbb{Z}^2 \to \mathbb{R},\ \mathbf{u} \mapsto x[\mathbf{u}]$. The action on the domain induces an action on such functions, the **regular representation** (see also Assignment 2), defined for every $g \in p4$ by

$$
[g.x](\mathbf{u}) \;:=\; x\bigl(g^{-1}.\mathbf{u}\bigr),
$$

where $g^{-1}.\mathbf{u}$ is the pixel of the original image $x$ that is moved to position $\mathbf{u}$ in the transformed image $g.x$.

In code we represent group elements as Python tuples `(k, (tx, ty))`. We split the work into two tasks: Task 3.1 isolates the $C_4$ index arithmetic, and Task 3.2 implements the $p4$ action `p4_act` and verifies the homomorphism property numerically.

(In Section 4 the equivariant CNN layers we'll build only need to bookkeep the $C_4$ part explicitly — the translations are handled silently by `F.conv2d`.)


<div class="alert alert-warning">
    <h3>Task 3.1: The cyclic group <span>$C_4$</span> [0.5 pt]</h3>
    Implement <code>c4_product(a, b)</code> and <code>c4_inverse(a)</code> where <code>a, b ∈ {0, 1, 2, 3}</code> are rotation indices.
</div>


In [ ]:
def c4_product(a, b):
    ### BEGIN SOLUTION

    ### END SOLUTION


def c4_inverse(a):
    ### BEGIN SOLUTION

    ### END SOLUTION


# Group axioms sanity check
for a in range(4):
    for b in range(4):
        assert c4_product(a, c4_inverse(a)) == 0
        assert c4_product(c4_product(a, b), c4_inverse(b)) == a
print("C4 group axioms verified.")


<div class="alert alert-warning">
    <h3>Task 3.2: Action of <span>$p4$</span> on images [0.5 pt]</h3>
    Implement <code>p4_act(g, x)</code> where <code>g = (k, (tx, ty))</code> with <code>k ∈ {0,1,2,3}</code> a rotation index and <code>(tx, ty)</code> an integer translation. <code>x</code> is an image tensor of shape <code>(..., H, W)</code>. Apply the rotation first and then a cyclic shift, using these conventions: the rotation turns the image <b>counter-clockwise</b> (the mathematically positive direction) by <code>k</code> steps of 90°, and the translation shifts it by <code>tx</code> along the width axis and <code>ty</code> along the height axis.
    <br><br>
    We will then run a homomorphism check, <code>p4_act(p4_product(g1, g2), x) == p4_act(g1, p4_act(g2, x))</code>, on several random group elements.
</div>


In [ ]:
def p4_product(g1, g2):
    '''Semidirect-product composition (k1, t1) · (k2, t2).'''
    k1, t1 = g1
    k2, t2 = g2
    # rotate t2 by k1. The pixel convention used by torch.rot90 + torch.roll
    # makes the corresponding rotation on the 2D translation vector
    rx, ry = t2
    for _ in range(k1):
        rx, ry = ry, -rx
    return ((k1 + k2) % 4, (t1[0] + rx, t1[1] + ry))


def p4_act(g, x):
    '''Apply g = (k, (tx, ty)) ∈ p4 to a batch of images of shape (..., H, W).'''
    ### BEGIN SOLUTION

    ### END SOLUTION


# Homomorphism check on a random image (provided).
torch.manual_seed(0)
x_test = torch.randn(1, 1, 16, 16)
rng = np.random.default_rng(0)
for _ in range(20):
    g1 = (int(rng.integers(0, 4)), (int(rng.integers(-3, 4)), int(rng.integers(-3, 4))))
    g2 = (int(rng.integers(0, 4)), (int(rng.integers(-3, 4)), int(rng.integers(-3, 4))))
    lhs = p4_act(p4_product(g1, g2), x_test)
    rhs = p4_act(g1, p4_act(g2, x_test))
    assert torch.allclose(lhs, rhs, atol=1e-5), "p4_act is not a group homomorphism!"
print("p4 action satisfies p4_act(g1·g2, x) == p4_act(g1, p4_act(g2, x)).")


# 4. Lifting and group convolution

We now design layers that are equivariant to the full $p4$ action. Since `F.conv2d` already gives us the $\mathbb{Z}^2$ part for free, the layer code only has to bookkeep the $C_4$ rotation index and the resulting network is $p4$-equivariant. There are exactly **two** kinds of layer:

### Lifting convolution
The first layer maps a 2D image (a function on $\mathbb{Z}^2$) to a function indexed by both a rotation $r^k \in C_4$ *and* a position $\mathbf{u} \in \mathbb{Z}^2$:

$$
[\, f \star_{\text{lift}} \psi \,](\,k, \mathbf{u}\,) \;=\; \sum_{\mathbf{v}} f(\mathbf{v}) \, \psi\bigl(r^{-k}(\mathbf{v} - \mathbf{u})\bigr).
$$

In practice we keep **one** learnable kernel $\psi$ and produce 4 output channels by convolving the input with the 4 rotated versions $\{\psi, r\psi, r^2\psi, r^3\psi\}$. The output of a lifting layer therefore has shape `(B, C_out, |C4|=4, H, W)`.

### Group convolution
Subsequent layers map (rotation × position)-indexed feature maps to feature maps of the same kind. The kernel is itself indexed by a rotation, with shape `(C_out, C_in, 4, k, k)`, and applying it requires both **rotating the kernel pixels** and **cyclically permuting its 4 input-rotation channels**:

$$
[\, f \star_G \psi \,](\,k, \mathbf{u}\,) \;=\; \sum_{j=0}^{3} \sum_{\mathbf{v}} f(j, \mathbf{v}) \, \psi\bigl(\,(k-j) \bmod 4,\; r^{-k}(\mathbf{v} - \mathbf{u})\bigr).
$$

Concretely, for output rotation index $k \in \{0,1,2,3\}$, the kernel applied to the lifted feature map is the original kernel rotated by $k$ in pixel space *and* rolled by $k$ along the input-rotation axis.


<div class="alert alert-warning">
    <h3>Task 4.1: <code>LiftingConv2d</code> [1 pt]</h3>
    Implement the lifting layer. It holds a single learnable weight <code>w</code> of shape <code>(C_out, C_in, k, k)</code> and a bias of shape <code>(C_out,)</code>. The <code>forward</code> maps an input <code>(B, C_in, H, W)</code> to a lifted feature of shape <code>(B, C_out, 4, H, W)</code>, where output rotation channel <code>r ∈ {0,1,2,3}</code> is the convolution (<code>padding=1</code>) of the input with <code>w</code> rotated by <code>r·90°</code>.
    <br><br>
    <i>Hint 1:</i> all four rotated kernels can be applied in a single <code>F.conv2d</code> call by stacking them along the output-channel axis.
    <br><br>
    <i>Hint 2:</i> as in Task 2.1, use a Kaiming-style uniform initialization again.
    <br><br>
    The cell below will then verify numerically that <code>LiftingConv2d</code> is $p4$-equivariant: rotating the input by 90° must equal rotating the output spatially by 90° <i>and</i> cyclically shifting its rotation channels by 1.
</div>


In [ ]:
class LiftingConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, padding=1):
        super().__init__()
        self.in_channels  = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.padding = padding
        ### BEGIN SOLUTION

        ### END SOLUTION

    def forward(self, x):
        # x: (B, C_in, H, W)  ->  (B, C_out, 4, H, W)
        ### BEGIN SOLUTION

        ### END SOLUTION


# Equivariance check.
torch.manual_seed(0)
lift = LiftingConv2d(1, 8).eval()
x_eq = torch.randn(1, 1, 16, 16)
with torch.no_grad():
    y = lift(x_eq)  # (1, 8, 4, 16, 16)
    y_rot = lift(torch.rot90(x_eq, k=1, dims=(-2, -1)))  # output of rotated input
    # Expected: spatially rotate y AND cyclically shift the group axis (dim=2) by +1.
    y_expected = torch.rot90(y, k=1, dims=(-2, -1)).roll(shifts=1, dims=2)
assert torch.allclose(y_rot, y_expected, atol=1e-5), "LiftingConv2d is NOT p4-equivariant."
print("LiftingConv2d is p4-equivariant up to numerical precision.")


<div class="alert alert-info">
    <h3>Visualize the lifted feature maps</h3>
    Below we show the 4 rotation channels for one output feature of <code>LiftingConv2d</code> applied to a real image. When the input is rotated by 90°, the same set of feature maps appears but cyclically shifted.
</div>


In [ ]:
img, _ = next(iter(test_loader))
img = img[:1]                                      # (1, 1, 28, 28)
img_rot = torch.rot90(img, k=1, dims=(-2, -1))
with torch.no_grad():
    feat       = lift(img)[0, 0]                  # (4, H, W) — channel 0
    feat_rot   = lift(img_rot)[0, 0]

imshow_grid(feat,     title="LiftingConv2d output (4 rotation channels)")
imshow_grid(feat_rot, title="Same after rotating input by 90° (channels shift cyclically)")


<div class="alert alert-warning">
    <h3>Task 4.2: <code>GroupConv2d</code> [1 pt]</h3>
    Implement the group-to-group convolution. The learnable weight has shape <code>(C_out, C_in, 4, k, k)</code> with a bias of shape <code>(C_out,)</code>. For each output rotation <code>r ∈ {0,1,2,3}</code> build the transformed weight by:
    <ol>
        <li>Rotating the kernel <b>pixels</b> by <code>r·90°</code>.</li>
        <li>Rolling the kernel along its <b>input-rotation axis</b> by <code>r</code>.</li>
    </ol>
    Stack the 4 transformed weights along a new leading axis, fold the input-rotation axis into channels (so the kernel becomes <code>(4*C_out, C_in*4, k, k)</code>), and call <code>F.conv2d</code> once on the input reshaped to <code>(B, C_in*4, H, W)</code>. Then reshape back to <code>(B, C_out, 4, H, W)</code>.
    <br><br>
    The cell below will then verify numerically that <code>GroupConv2d</code> is $p4$-equivariant.
</div>


In [ ]:
class GroupConv2d(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size=3, padding=1):
        super().__init__()
        self.in_channels  = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size
        self.padding = padding
        ### BEGIN SOLUTION

        ### END SOLUTION

    def forward(self, x):
        # x: (B, C_in, 4, H, W)  ->  (B, C_out, 4, H, W)
        ### BEGIN SOLUTION

        ### END SOLUTION


# Equivariance check: feed the lifted feature, rotate input by 90°, compare.
torch.manual_seed(0)
gconv = GroupConv2d(8, 16).eval()
with torch.no_grad():
    feat_lifted = lift(x_eq)  # (1, 8, 4, 16, 16)
    feat_lifted_rot = torch.rot90(feat_lifted, k=1, dims=(-2, -1)).roll(shifts=1, dims=2)
    out = gconv(feat_lifted)
    out_rot = gconv(feat_lifted_rot)
    expected = torch.rot90(out, k=1, dims=(-2, -1)).roll(shifts=1, dims=2)
assert torch.allclose(out_rot, expected, atol=1e-5), "GroupConv2d is NOT p4-equivariant."
print("GroupConv2d is p4-equivariant up to numerical precision.")


# 5. The $p4$-CNN

We now assemble a full classifier. Pointwise nonlinearities (ReLU) commute with the cyclic shift on the rotation axis, so they preserve $p4$-equivariance. Max-pooling over only the **spatial** dimensions also preserves it. The final step is to make the network *invariant*, not just equivariant: we pool over both spatial dimensions *and* the rotation axis before the linear classifier. Both `mean` and `max` over those axes are invariant to the $p4$ action (a cyclic shift of the group axis plus a spatial rotation just permutes the values being reduced), so we concatenate the two — a richer readout than `mean` alone, still fully invariant.


<div class="alert alert-warning">
    <h3>Task 5.1: <code>P4CNN</code> [1 pt]</h3>
    Build the architecture below and train it for 15 epochs:
    <ul>
        <li><code>LiftingConv2d(1 → 24)</code> + ReLU + 2×2 spatial max-pool</li>
        <li><code>GroupConv2d(24 → 48)</code> + ReLU + 2×2 spatial max-pool</li>
        <li><code>GroupConv2d(48 → 48)</code> + ReLU</li>
        <li>Global readout: concatenate the <b>mean</b> and the <b>max</b> over <b>(group, H, W)</b> dims, then <code>Linear(96 → 10)</code></li>
    </ul>
    <i>Tip:</i> the lifted feature has shape <code>(B, C, 4, H, W)</code>. <code>F.max_pool2d</code> only handles 4D tensors, so use <code>F.max_pool3d</code> with <code>kernel_size=(1, 2, 2)</code>.
</div>


In [ ]:
class P4CNN(nn.Module):
    def __init__(self):
        super().__init__()
        ### BEGIN SOLUTION

        ### END SOLUTION

    def forward(self, x):
        ### BEGIN SOLUTION

        ### END SOLUTION


p4cnn = P4CNN()
print(f"P4CNN params: {sum(p.numel() for p in p4cnn.parameters()):,}")

print("\nTraining P4CNN ...")
acc_p4cnn = train_model(p4cnn, train_loader, test_loader, epochs=15)


<div class="alert alert-info">
    <h3>Invariance of the $p4$-CNN</h3>
    We probe the trained model with the same angles as before. Because $p4$ only contains 90° rotations (its rotation part is $C_4$), we expect <b>essentially zero</b> invariance error at every tested angle.
</div>


In [ ]:
err_p4cnn = [invariance_error(p4cnn, x_probe, a) for a in ANGLES]
for a, e in zip(ANGLES, err_p4cnn):
    print(f"P4CNN | angle {a:3d}° | max output deviation = {e:.3f}")


# 6. Putting it all together

We now compare all three models on the two axes we care about: **test accuracy** on rotated FashionMNIST and **invariance error** as a function of rotation angle.


<div class="alert alert-info">
    <h3>Plots</h3>
    Nothing to do — just run the cell.
</div>


In [ ]:
fig, (ax_acc, ax_err) = plt.subplots(1, 2, figsize=(11, 4))

names = ["VanillaCNN", "IsotropicCNN", "P4CNN"]
accs  = [acc_vanilla, acc_iso, acc_p4cnn]
colors = ["#888888", "#1f77b4", "#d62728"]

ax_acc.bar(names, accs, color=colors)
ax_acc.set_ylim(0, 1.0)
ax_acc.set_ylabel("test accuracy")
ax_acc.set_title("Accuracy on rotated FashionMNIST")
for i, v in enumerate(accs):
    ax_acc.text(i, v + 0.02, f"{v:.3f}", ha="center")

ax_err.plot(ANGLES, err_vanilla, "o-", color=colors[0], label=names[0])
ax_err.plot(ANGLES, err_iso,     "s-", color=colors[1], label=names[1])
ax_err.plot(ANGLES, err_p4cnn,   "^-", color=colors[2], label=names[2])
ax_err.set_xlabel("rotation angle (degrees)")
ax_err.set_ylabel("max output deviation")
ax_err.set_title("Invariance error vs rotation angle")
ax_err.grid(True, alpha=0.3)
ax_err.legend()

plt.tight_layout()
plt.show()


<div class="alert alert-warning">
    <h3>Task 6.1: Reading the plots [1 pt]</h3>
    The three models occupy three different points on the <b>accuracy / equivariance / expressivity</b> trade-off. Describe in 3–4 sentences:
    <ol>
        <li>The ranking by test accuracy.</li>
        <li>The ranking by invariance error at 90°.</li>
        <li>What this tells us about the gap between <i>per-filter invariance</i> (isotropic) and <i>architectural equivariance</i> ($p4$).</li>
    </ol>
</div>


### Interpretation

...
